# Crypto.Net · 04 — Solana

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./04-solana.en.ipynb)

Akun yang kompatibel dengan Phantom, format kunci, pembacaan live devnet, program-derived address dan transaksi yang disimulasikan dengan instruksi Compute Budget, System dan Memo.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Wallet;
using Crypto.Net.Solana;

var wallet  = HdWallet.Generate();
var account = wallet.GetSolanaAccount(0, SolanaChain.Devnet);   // m/44'/501'/0'/0' (Phantom, Solflare)
Console.WriteLine($"{account.Address} ({account.DerivationPath})");
Console.WriteLine($"Phantom secret key format: {account.ExportBase58SecretKey()[..12]}… (64 bytes)");
Console.WriteLine($"solana-cli keypair JSON:   {account.ExportKeypairJson()[..40]}…");

## Devnet live

In [ ]:
var rpc = new SolanaRpcClient(SolanaChain.Devnet);
var (blockhash, lastValid) = await rpc.GetLatestBlockhashAsync();
Console.WriteLine($"slot {await rpc.GetBlockNumberAsync():N0}, blockhash {blockhash} (valid until height {lastValid:N0})");
Console.WriteLine($"balance {await rpc.GetBalanceAsync(account.Address)} SOL");
Console.WriteLine($"rent-exempt minimum for 165 bytes: {Amount.FromLamports(await rpc.GetMinimumBalanceForRentExemptionAsync(165))} SOL");

## Program-derived address

PDA berada di luar kurva ed25519, sehingga tidak ada private key untuknya. Associated token account (ATA) adalah PDA yang dipakai setiap dompet untuk mint tertentu (di sini: USDC devnet).

In [ ]:
const string DevnetUsdc = "4zMMC9srt5Ri5X14GAgXhaHii3GnPAEERYPJgZJDncDU";
string ata = SolanaAddress.GetAssociatedTokenAddress(account.Address.Value, DevnetUsdc);
var (vault, bump) = SolanaAddress.FindProgramAddress([SolanaAddress.Utf8Seed("vault"), SolanaAddress.Decode(account.Address.Value)], SolanaAddress.MemoProgram);

new[]
{
    new { What = "wallet", Address = account.Address.Value, OnCurve = SolanaAddress.IsOnCurve(account.Address.Value) },
    new { What = "USDC ATA", Address = ata, OnCurve = SolanaAddress.IsOnCurve(ata) },
    new { What = $"PDA (bump {bump})", Address = vault, OnCurve = SolanaAddress.IsOnCurve(vault) },
}

## Membangun, menandatangani dan mensimulasikan

Akun baru belum punya SOL, sehingga simulasi melaporkan error — itu adalah node yang memvalidasi transaksi bertanda tangan kita. Isi saldo dengan `RequestAirdropAsync` (devnet) untuk melihatnya berhasil.

In [ ]:
var tx = new SolanaTransaction(account.Address.Value, blockhash)
    .Add(ComputeBudgetProgram.SetComputeUnitPrice(1_000))
    .Add(SystemProgram.Transfer(account.Address.Value, "11111111111111111111111111111112", 5_000))
    .Add(MemoProgram.Memo("Hello from Crypto.Net", account.Address.Value));
account.Sign(tx);

byte[] wire = tx.Serialize();
Console.WriteLine($"signature (tx id) {tx.Signature}");
Console.WriteLine($"{wire.Length} bytes · accounts: {string.Join(", ", tx.CompileAccounts().Select(a => a.PublicKey[..6] + (a.IsSigner ? "(s)" : "") + (a.IsWritable ? "(w)" : "")))}");

var (error, logs) = await rpc.SimulateTransactionAsync(wire);
Console.WriteLine(error is null ? "simulation succeeded" : $"simulation error: {error}");
foreach (var line in logs) Console.WriteLine($"  {line}");

## Mengirim sungguhan

```csharp
await rpc.RequestAirdropAsync(account.Address.Value, 1_000_000_000); // devnet, dibatasi laju
TxHash sig = await rpc.TransferAsync(account, "<penerima>", Amount.FromSol(0.1m), priorityFeeMicroLamports: 1_000);
TxHash spl = await rpc.TransferTokenAsync(account, DevnetUsdc, "<penerima>", Amount.Parse("1", 6));
```

In [ ]:
account.Dispose(); rpc.Dispose(); wallet.Dispose();
"disposed"